# Task 5 — Step 4: Finalize Web-Retrieval Augmented Training Sets

Loads `web_labeled.jsonl` (Step 3) for both datasets, filters to `relevant=true` pairs, and
reports the three dataset-quality layers by:

- **Label ratio** (target ~1:3 pos/neg)
- **Corner-case proportion** (token-Jaccard hard positives/negatives, same definition as
  `02_wdc-products_analysis.ipynb` / `03_dblp-scholar_analysis.ipynb` for cross-strategy comparability)
- **Decisive attribute coverage** (WDC Products only — color, memory, size, bundle, edition,
  model number)
- **Entity/source diversity** (bucket distribution from Step 1 sampling)
- **Spot-check** of a random sample for label-noise review

Then writes `train_aug_web.txt` = `train.txt` + all usable web-retrieved pairs, for both
datasets.

## Imports & helpers

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("../").resolve()
PROCESSED = ROOT / "data" / "processed"

_COL_VAL_RE = re.compile(r"COL\s+\S+\s+VAL\s*")


def value_tokens(col_val_str: str) -> set[str]:
    """Strip COL/VAL markers and return lowercased value token set."""
    text = _COL_VAL_RE.sub(" ", col_val_str)
    return set(text.lower().split())


def jaccard(a: set, b: set) -> float:
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)


def pair_sim(left: str, right: str) -> float:
    return jaccard(value_tokens(left), value_tokens(right))


def trunc(s: str, n: int = 90) -> str:
    vals = _COL_VAL_RE.sub(" ", str(s)).strip()
    return vals[:n] + "…" if len(vals) > n else vals


# Decisive attribute patterns (WDC Products only) — same as 02_wdc-products_analysis.ipynb
ATTR_PATTERNS = {
    "color": re.compile(r"\b(black|white|red|silver|blue|gold|grey|gray|green|yellow|pink)\b", re.I),
    "memory/storage": re.compile(r"\d+\s*(gb|tb|mb)", re.I),
    "size": re.compile(r'\d+\s*(inch|in\b|"|cm|mm|\bl\b|xl\b|xs\b|sm\b)', re.I),
    "bundle": re.compile(r"\b(bundle|kit|set|pack)\b", re.I),
    "edition": re.compile(r"\b(edition|version|\bse\b|\ble\b|\bpro\b|\bplus\b)\b", re.I),
    "model number": re.compile(r"\b[A-Z]{1,4}\d{3,}\b"),
}

print("Helpers defined.")

Helpers defined.


## Per-dataset report

In [2]:
DATASETS = ["wdc-products", "dblp-scholar"]

reports = {}  # dataset -> dict of computed stats (used later for train_aug_web.txt writing)

for dataset in DATASETS:
    print(f"\n{'='*70}")
    print(f"  {dataset}")
    print(f"{'='*70}")

    out_dir = PROCESSED / dataset
    all_records = [json.loads(l) for l in open(out_dir / "web_labeled.jsonl")]
    usable = [r for r in all_records if r["relevant"]]

    n_pos = sum(1 for r in usable if r["label"] == 1)
    n_neg = len(usable) - n_pos
    pos_rate = n_pos / len(usable) * 100 if usable else 0

    print(f"\nWeb-retrieved pairs:")
    print(f"  Candidates processed : {len(all_records)}")
    print(f"  Usable (relevant)    : {len(usable)}  ({len(usable)/len(all_records)*100:.1f}%)")
    print(f"  Matches (label=1)    : {n_pos}  ({pos_rate:.1f}%)")
    print(f"  Non-matches (label=0): {n_neg}  ({100-pos_rate:.1f}%)")
    target_status = "within 1:3 target (~25% pos)" if 15 <= pos_rate <= 35 else "outside 1:3 target (~25% pos)"
    print(f"  Ratio check          : {target_status}")

    # --- Corner cases (token-Jaccard, same thresholds as 02/03_analysis.ipynb) ---
    sims = [pair_sim(r["left_text"], r["right_text"]) for r in usable]
    hard_pos = sum(1 for r, s in zip(usable, sims) if r["label"] == 1 and s < 0.3)
    hard_neg = sum(1 for r, s in zip(usable, sims) if r["label"] == 0 and s > 0.4)
    n_cc = hard_pos + hard_neg

    print(f"\nCorner cases among web-retrieved pairs (token-Jaccard thresholds):")
    print(f"  Hard positives (label=1, Jaccard<0.3): {hard_pos}  ({100*hard_pos/len(usable):.1f}%)")
    print(f"  Hard negatives (label=0, Jaccard>0.4): {hard_neg}  ({100*hard_neg/len(usable):.1f}%)")
    print(f"  Total corner cases                   : {n_cc}  ({100*n_cc/len(usable):.1f}%)")
    cc_status = "within 40-50% target" if 0.40 <= n_cc/len(usable) <= 0.50 else "outside 40-50% target"
    print(f"  Target: 40-50% -> {cc_status}")

    # --- Decisive attribute coverage (WDC only) ---
    if dataset == "wdc-products":
        print(f"\nDecisive attribute coverage (% of web-retrieved pairs mentioning each category):")
        for attr, pat in ATTR_PATTERNS.items():
            hits = sum(1 for r in usable if pat.search(r["left_text"]) or pat.search(r["right_text"]))
            pct = 100 * hits / len(usable)
            flag = "low" if pct < 10 else ""
            print(f"  {attr:<18}: {hits:4d} pairs  ({pct:5.1f}%)  {flag}")

    # --- Diversity (bucket distribution from Step 1) ---
    web_query = pd.read_csv(out_dir / "web_query_entities.csv")
    n_buckets = web_query["bucket"].nunique()
    top_buckets = web_query["bucket"].value_counts().head(5)
    print(f"\nEntity diversity (Step 1 sample): {len(web_query)} entities across {n_buckets} distinct buckets")
    print(f"  Top buckets:\n{top_buckets.to_string()}")

    reports[dataset] = {
        "usable": usable,
        "sims": sims,
        "n_pos": n_pos,
        "n_neg": n_neg,
    }


  wdc-products

Web-retrieved pairs:
  Candidates processed : 900
  Usable (relevant)    : 803  (89.2%)
  Matches (label=1)    : 649  (80.8%)
  Non-matches (label=0): 154  (19.2%)
  Ratio check          : outside 1:3 target (~25% pos)

Corner cases among web-retrieved pairs (token-Jaccard thresholds):
  Hard positives (label=1, Jaccard<0.3): 545  (67.9%)
  Hard negatives (label=0, Jaccard>0.4): 1  (0.1%)
  Total corner cases                   : 546  (68.0%)
  Target: 40-50% -> outside 40-50% target

Decisive attribute coverage (% of web-retrieved pairs mentioning each category):
  color             :  246 pairs  ( 30.6%)  
  memory/storage    :  239 pairs  ( 29.8%)  
  size              :  234 pairs  ( 29.1%)  
  bundle            :  101 pairs  ( 12.6%)  
  edition           :  123 pairs  ( 15.3%)  
  model number      :  222 pairs  ( 27.6%)  

Entity diversity (Step 1 sample): 300 entities across 97 distinct buckets
  Top buckets:
bucket
_none            20
Epson            17
Corsai

## Spot-check: random sample for label-noise review

In [3]:
rng = np.random.default_rng(42)

for dataset in DATASETS:
    usable = reports[dataset]["usable"]
    sample_idx = rng.choice(len(usable), size=min(10, len(usable)), replace=False)

    print(f"\n{'='*70}")
    print(f"  {dataset} — random spot-check (n={len(sample_idx)})")
    print(f"{'='*70}")
    for i in sample_idx:
        r = usable[i]
        print(f"  label={r['label']}  sim={reports[dataset]['sims'][i]:.3f}")
        print(f"  LEFT : {trunc(r['left_text'])}")
        print(f"  RIGHT: {trunc(r['right_text'])}")
        print(f"  reason: {r['reasoning']}")
        print()


  wdc-products — random spot-check (n=10)
  label=1  sim=0.125
  LEFT : Epson  Epson Tinte matte schwarz 700ml f. 7900/9900   258.32  EUR
  RIGHT: Epson  Epson UltraChrome HDR Ink Matte Black 700ml for Stylus Pro 7700, 7890, 7900, 9700, …
  reason: Both records describe the same Epson matte black ink cartridge 700ml compatible with the 7900/9900 printers (T636800).

  label=1  sim=0.230
  LEFT : Daniel Wellington  Daniel Wellington Classic Sheffield 36mm Unisex Watch, Japanese Quartz …
  RIGHT: Daniel Wellington  Daniel Wellington Unisex Classic Sheffield 36mm Rose Gold Black DW00100…
  reason: The Amazon listing matches the original record: same brand (Daniel Wellington), same model (Classic Sheffield 36mm), same model number (0508DW), same rose gold/leather strap configuration, confirming it is the same product.

  label=1  sim=0.065
  LEFT : sandisk  64GB SanDisk Extreme micro SD XC Memory Card V-Class 30 U3 4K Video A1 100MB/s  S…
  RIGHT: SanDisk  SanDisk 64GB Extreme microSDXC V

## Write `train_aug_web.txt`

In [4]:
for dataset in DATASETS:
    out_dir = PROCESSED / dataset
    usable = reports[dataset]["usable"]
    train_path = out_dir / "train.txt"
    aug_path = out_dir / "train_aug_web.txt"

    # Existing train pairs
    existing_lines = open(train_path).readlines()
    n_existing = len(existing_lines)
    n_existing_pos = sum(1 for l in existing_lines if l.rstrip("\n").split("\t")[2] == "1")

    with open(aug_path, "w") as f:
        for line in existing_lines:
            f.write(line)
        for r in usable:
            f.write(f"{r['left_text']}\t{r['right_text']}\t{r['label']}\n")

    n_total = n_existing + len(usable)
    n_total_pos = n_existing_pos + reports[dataset]["n_pos"]

    print(f"\n{dataset}:")
    print(f"  {train_path.name}     : {n_existing:,} pairs  ({n_existing_pos:,} pos, {100*n_existing_pos/n_existing:.1f}%)")
    print(f"  + web-retrieved      : {len(usable):,} pairs  ({reports[dataset]['n_pos']:,} pos, "
          f"{100*reports[dataset]['n_pos']/len(usable):.1f}%)")
    print(f"  = {aug_path.name} : {n_total:,} pairs  ({n_total_pos:,} pos, {100*n_total_pos/n_total:.1f}%)")


wdc-products:
  train.txt     : 2,500 pairs  (500 pos, 20.0%)
  + web-retrieved      : 803 pairs  (649 pos, 80.8%)
  = train_aug_web.txt : 3,303 pairs  (1,149 pos, 34.8%)

dblp-scholar:
  train.txt     : 17,223 pairs  (3,207 pos, 18.6%)
  + web-retrieved      : 742 pairs  (640 pos, 86.3%)
  = train_aug_web.txt : 17,965 pairs  (3,847 pos, 21.4%)
